# 🔬 PSIE — Programme Structural Intelligence Engine
## Oman Vision 2040 — Interactive Demo

This notebook demonstrates the Programme Structural Intelligence Engine (PSIE)
using the Oman Vision 2040 national programme as a case study.

**What this does:**
1. Loads a major programme encoded as a typed knowledge graph (23 nodes, 35 edges)
2. Runs the three S³ analytical lenses (Scoping, Scaffolding, Sensing)
3. Simulates disruption cascades through the dependency network
4. Generates publication-quality visualisations

**Author:** Bruno Chikeka — Saïd Business School, University of Oxford  
**Supervisor:** Prof. Daniel Armanios

## 1. Setup

In [ ]:
# Clone the repo (Colab only)
!git clone https://github.com/chiKeka/psie.git 2>/dev/null || echo 'Repo already cloned'
%cd psie
!pip install -q networkx numpy matplotlib

In [ ]:
import sys, os
sys.path.insert(0, '.')

from psie.graph_schema import ProgrammeGraph
from psie.propagation import CascadeEngine, Disruption, DisruptionType
from psie.s3_metrics import S3Analyser
from psie.visualization import (
    plot_programme_graph,
    plot_vulnerability_ranking,
    plot_cascade_timeline,
)
import json
import matplotlib.pyplot as plt
%matplotlib inline

print('PSIE loaded successfully ✓')

## 2. Load the Programme Graph

In [ ]:
pg = ProgrammeGraph.from_json('data/oman_vision_2040.json')
print(pg.summary())

In [ ]:
# Visualise the full programme graph
fig = plot_programme_graph(pg, title='PSIE — Oman Vision 2040', figsize=(14, 10), show=True)

## 3. S³ Lens 1 — Scoping (Vulnerability Analysis)

*What does the programme depend on? Where is it most vulnerable?*

The scoping lens computes a composite vulnerability score combining degree centrality,
node vulnerability, inverse absorptive capacity, and criticality.

In [ ]:
analyser = S3Analyser(pg)
scoping = analyser.scoping()

print('Top 10 by Composite Vulnerability Score:')
print('-' * 60)
for nid, score in scoping['vulnerability_ranking'][:10]:
    d = pg.G.nodes[nid]
    print(f"  {d['label']:40s}  {score:.4f}  [{d['family']}]")

print(f"\nFragility Concentration (Gini): {scoping['fragility_concentration_gini']:.4f}")
print(f"Minimum Viable System: {len(scoping['minimum_viable_system'])} nodes")

In [ ]:
fig = plot_vulnerability_ranking(scoping['vulnerability_ranking'], pg, top_n=15, show=True)

## 4. S³ Lens 2 — Scaffolding (Information Flow)

*How does the programme hold together? Where are the bottlenecks?*

In [ ]:
scaffolding = analyser.scaffolding()

print('Information Bottlenecks:')
for nid, score in scaffolding['information_bottlenecks']:
    d = pg.G.nodes[nid]
    print(f"  {d['label']:40s}  betweenness={score:.4f}")

print(f"\nAvg Path Length: {scaffolding['flow_efficiency_avg_path_length']:.4f}")

if scaffolding['scaffolding_gaps']:
    print('\nScaffolding Gaps (high criticality, low support):')
    for gap in scaffolding['scaffolding_gaps'][:5]:
        print(f"  {gap['label']:40s}  crit={gap['criticality']:.1f}  "
              f"support={gap['support_edges']}  severity={gap['gap_severity']:.4f}")

## 5. S³ Lens 3 — Sensing (Gap Detection)

*Where should attention be directed? What blind spots exist?*

In [ ]:
sensing = analyser.sensing()

print('Orphan Nodes (potential blind spots):')
for orphan in sensing['orphan_nodes']:
    print(f"  {orphan['label']:40s}  degree={orphan['degree']}  [{orphan['family']}]")

print(f"\nAttention Concentration: {sensing['attention_concentration']}")
if sensing['attention_bias_warning']:
    print('⚠️  ATTENTION BIAS WARNING: leadership focus is concentrated in one family')

## 6. Disruption Simulation (Attack Mode)

*If a disruption hits, what breaks? How far does the cascade travel?*

We simulate four disruption scenarios from the CMR paper analysis.

In [ ]:
# Load disruption scenarios from the data file
with open('data/oman_vision_2040.json') as f:
    data = json.load(f)

engine = CascadeEngine(pg)
results = []

for s in data['disruption_scenarios']:
    d = Disruption(
        id=s['id'], label=s['label'],
        disruption_type=DisruptionType(s['disruption_type']),
        injection_node=s['injection_node'],
        magnitude=s['magnitude'],
    )
    result = engine.simulate(d)
    results.append(result)
    print(result.summary())
    print()

In [ ]:
# Visualise the widest cascade
widest = max(results, key=lambda r: r.total_nodes_affected)
print(f'Widest cascade: {widest.disruption_label}')

fig = plot_cascade_timeline(widest, show=True)

In [ ]:
# Programme graph with cascade overlay
fig = plot_programme_graph(
    pg, cascade=widest,
    title=f'Cascade: {widest.disruption_label}',
    figsize=(14, 10), show=True
)

## 7. Your Own Disruption

Try injecting a custom disruption into any node:

In [ ]:
# Custom disruption — modify these values!
custom = Disruption(
    id='custom_1',
    label='Custom: Education System Collapse',
    disruption_type=DisruptionType.DELAY,
    injection_node='education',  # Try: 'governance', 'infrastructure', 'oil_gas'
    magnitude=0.9,
)

result = engine.simulate(custom)
print(result.summary())
print()

fig = plot_programme_graph(
    pg, cascade=result,
    title=f'Custom Cascade: {custom.label}',
    figsize=(14, 10), show=True
)

---

## Next Steps

This MVP demonstrates Layers 1–2 of the six-layer architecture. The full research programme will add:

- **Layer 3:** System dynamics temporal model (feedback loops, time delays)
- **Layer 4:** Bayesian prediction engine (probabilistic outcome forecasting)
- **Layer 5:** Reference-class calibration (empirical grounding)
- **Layer 6:** S³ interpretation layer (programme-management vocabulary)

**Seeking collaborators** in network science, system dynamics, Bayesian inference, and programme management.  
Contact: Bruno Chikeka — Saïd Business School, University of Oxford